In [2]:
from beir import util
import pathlib

dataset = "scifact"

url = f"https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/{dataset}.zip"

out_dir = pathlib.Path("DATA/evaluation")

data_path = util.download_and_unzip(
    url,
    out_dir
)

print(data_path)

DATA/evaluation/scifact


/Users/divyanshjain/Documents/Learning/mock_projects/ai_projects/rag/myenv/lib/python3.12/site-packages/beir/util.py:11: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm


In [3]:
from pathlib import Path

scifact_dir = Path(data_path)

for path in scifact_dir.rglob("*"):
    print(path)

DATA/evaluation/scifact/qrels
DATA/evaluation/scifact/corpus.jsonl
DATA/evaluation/scifact/queries.jsonl
DATA/evaluation/scifact/qrels/train.tsv
DATA/evaluation/scifact/qrels/test.tsv


In [4]:
from beir.datasets.data_loader import GenericDataLoader

corpus, queries, qrels = GenericDataLoader(
    data_folder=str(scifact_dir)
).load(split="test")

100%|██████████| 5183/5183 [00:00<00:00, 63572.36it/s]


In [5]:
print("Documents:", len(corpus))
print("Queries:", len(queries))
print("Qrels:", len(qrels))

Documents: 5183
Queries: 300
Qrels: 300


In [6]:
print("CORPUS")
print(next(iter(corpus.items())))

print("\nQUERY")
print(next(iter(queries.items())))

print("\nQREL")
print(next(iter(qrels.items())))

CORPUS
('4983', {'text': 'Alterations of the architecture of cerebral white matter in the developing human brain can affect cortical development and result in functional disabilities. A line scan diffusion-weighted magnetic resonance imaging (MRI) sequence with diffusion tensor analysis was applied to measure the apparent diffusion coefficient, to calculate relative anisotropy, and to delineate three-dimensional fiber architecture in cerebral white matter in preterm (n = 17) and full-term infants (n = 7). To assess effects of prematurity on cerebral white matter development, early gestation preterm infants (n = 10) were studied a second time at term. In the central white matter the mean apparent diffusion coefficient at 28 wk was high, 1.8 microm2/ms, and decreased toward term to 1.2 microm2/ms. In the posterior limb of the internal capsule, the mean apparent diffusion coefficients at both times were similar (1.2 versus 1.1 microm2/ms). Relative anisotropy was higher the closer birth w

In [7]:
import os
from qdrant_client import QdrantClient

client = QdrantClient(
    url=os.getenv("QDRANT_CLUSTER_ENDPOINT"),
    api_key=os.getenv("QDRANT_API_KEY"),
)

'https://10d22751-c9d1-4106-81de-93303760d055.australia-southeast1-0.gcp.cloud.qdrant.io'

In [ ]:
from qdrant_client import models

COLLECTION_NAME = "scifact_bm25"

client.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config={},
    sparse_vectors_config={
        "bm25": models.SparseVectorParams(
            modifier=models.Modifier.IDF
        )
    },
)

True

In [23]:
import uuid
from qdrant_client.models import PointStruct, Document

BATCH_SIZE = 100

items = list(corpus.items())

for start in range(0, len(items), BATCH_SIZE):

    batch = items[start:start + BATCH_SIZE]

    points = []

    for doc_id, doc in batch:
        points.append(
            PointStruct(
                id=str(uuid.uuid4()),
                vector={
                    "bm25": Document(
                        text=doc["title"] + "\n" + doc["text"],
                        model="qdrant/bm25",
                    )
                },
                payload={
                    "document_id": doc_id,
                    "title": doc["title"],
                    "text": doc["text"],
                },
            )
        )

    client.upsert(
        collection_name=COLLECTION_NAME,
        points=points,
    )

    print(f"Uploaded {min(start + BATCH_SIZE, len(items))}/{len(items)}")

Uploaded 100/5183
Uploaded 200/5183
Uploaded 300/5183
Uploaded 400/5183
Uploaded 500/5183
Uploaded 600/5183
Uploaded 700/5183
Uploaded 800/5183
Uploaded 900/5183
Uploaded 1000/5183
Uploaded 1100/5183
Uploaded 1200/5183
Uploaded 1300/5183
Uploaded 1400/5183
Uploaded 1500/5183
Uploaded 1600/5183
Uploaded 1700/5183
Uploaded 1800/5183
Uploaded 1900/5183
Uploaded 2000/5183
Uploaded 2100/5183
Uploaded 2200/5183
Uploaded 2300/5183
Uploaded 2400/5183
Uploaded 2500/5183
Uploaded 2600/5183
Uploaded 2700/5183
Uploaded 2800/5183
Uploaded 2900/5183
Uploaded 3000/5183
Uploaded 3100/5183
Uploaded 3200/5183
Uploaded 3300/5183
Uploaded 3400/5183
Uploaded 3500/5183
Uploaded 3600/5183
Uploaded 3700/5183
Uploaded 3800/5183
Uploaded 3900/5183
Uploaded 4000/5183
Uploaded 4100/5183
Uploaded 4200/5183
Uploaded 4300/5183
Uploaded 4400/5183
Uploaded 4500/5183
Uploaded 4600/5183
Uploaded 4700/5183
Uploaded 4800/5183
Uploaded 4900/5183
Uploaded 5000/5183
Uploaded 5100/5183
Uploaded 5183/5183


In [30]:
client.count(
    collection_name=COLLECTION_NAME
).count

5183

In [24]:
query_id, query_text = next(iter(queries.items()))

print(query_text)

0-dimensional biomaterials show inductive properties.


In [40]:
results = client.query_points(
    collection_name=COLLECTION_NAME,
    query=Document(
        text=query_text,
        model="qdrant/bm25",
    ),
    using="bm25",
    limit=100,
    with_payload=True,
).points

In [35]:
for result in results:
    print("=" * 80)
    print("Score:", result.score)
    print("ID:", result.id)
    print("Title:", result.payload["title"])
    print(result.payload["text"][:500])

Score: 11.388737
ID: c62b3f1f-05f0-4bfa-9389-9b9343de150f
Title: The first gene of the Bacillus subtilis clpC operon, ctsR, encodes a negative regulator of its own operon and other class III heat shock genes.
The Bacillus subtilis clpC operon is regulated by two stress induction pathways relying on either sigmaB or a class III stress induction mechanism acting at a sigmaA-like promoter. When the clpC operon was placed under the control of the isopropyl-beta-D-thiogalactopyranoside (IPTG)-inducible Pspac promoter, dramatic repression of the natural clpC promoters fused to a lacZ reporter gene was noticed after IPTG induction. This result strongly indicated negative regulation of the clpC operon by one
Score: 11.194368
ID: b36c58e2-1041-4206-9733-8163bc3e3a81
Title: Epithelial and mesenchymal subpopulations within normal basal breast cell lines exhibit distinct stem cell/progenitor properties.
It has been proposed that epithelial-mesenchymal transition (EMT) in mammary epithelial cells a

In [41]:
relevant_docs = qrels.get(query_id, {})

print("Relevant documents:", relevant_docs)

retrieved_ids = [
    result.payload["document_id"]
    for result in results
]

print("Retrieved:", retrieved_ids)

print(
    "Relevant retrieved:",
    set(retrieved_ids) & set(relevant_docs.keys())
)

Relevant documents: {'31715818': 1}
Retrieved: ['34386619', '43385013', '18953920', '10608397', '21257564', '40212412', '3770726', '27049238', '6550579', '13231899', '4983', '22478394', '7581911', '10582939', '31543713', '16532419', '36480032', '20155713', '393001', '26071782', '16939583', '803312', '44947611', '42421723', '38037690', '10086360', '1771079', '24660385', '10906636', '38944245', '1996292', '3566945', '27772649', '42373087', '1156322', '1346695', '11862753', '21377587', '10607877', '24555878', '21456232', '12658073', '26047921', '14827874', '11141995', '95764370', '59453688', '4422734', '31439189', '25298276', '12824568', '1944452', '25404036', '38805486', '40584205', '25747721', '7734150', '25301182', '6863070', '27396415', '1285713', '42240424', '32001951', '26990001', '1203035', '26025820', '26083387', '9580772', '121581019', '24928817', '13903052', '41928290', '825728', '18126445', '10074251', '10029970', '19651306', '6227220', '16287725', '17518195', '26886351', '1827

In [42]:
from qdrant_client.models import Document

def bm25_search(query_text, top_k=10):
    results = client.query_points(
        collection_name=COLLECTION_NAME,
        query=Document(
            text=query_text,
            model="qdrant/bm25",
        ),
        using="bm25",
        limit=top_k,
        with_payload=True,
    ).points

    return [
        {
            "document_id": result.payload["document_id"],
            "score": result.score,
        }
        for result in results
    ]

In [43]:
import math


def recall_at_k(retrieved, relevant, k):
    retrieved_k = set(retrieved[:k])
    relevant = set(relevant)

    if not relevant:
        return 0.0

    return len(retrieved_k & relevant) / len(relevant)


def reciprocal_rank(retrieved, relevant, k):
    relevant = set(relevant)

    for rank, doc_id in enumerate(retrieved[:k], start=1):
        if doc_id in relevant:
            return 1 / rank

    return 0.0


def ndcg_at_k(retrieved, relevant, k):
    relevant = set(relevant)

    dcg = 0.0

    for rank, doc_id in enumerate(retrieved[:k], start=1):
        if doc_id in relevant:
            dcg += 1 / math.log2(rank + 1)

    ideal_hits = min(len(relevant), k)

    if ideal_hits == 0:
        return 0.0

    idcg = sum(
        1 / math.log2(rank + 1)
        for rank in range(1, ideal_hits + 1)
    )

    return dcg / idcg

In [44]:
evaluation_results = []

for i, (query_id, query_text) in enumerate(queries.items()):

    relevant_docs = [
        doc_id
        for doc_id, relevance in qrels.get(query_id, {}).items()
        if relevance > 0
    ]

    if not relevant_docs:
        continue

    results = bm25_search(query_text, top_k=10)

    retrieved_docs = [
        result["document_id"]
        for result in results
    ]

    evaluation_results.append({
        "query_id": query_id,
        "recall@5": recall_at_k(
            retrieved_docs,
            relevant_docs,
            5,
        ),
        "recall@10": recall_at_k(
            retrieved_docs,
            relevant_docs,
            10,
        ),
        "mrr@10": reciprocal_rank(
            retrieved_docs,
            relevant_docs,
            10,
        ),
        "ndcg@10": ndcg_at_k(
            retrieved_docs,
            relevant_docs,
            10,
        ),
    })

    if (i + 1) % 50 == 0:
        print(f"Processed {i + 1}/{len(queries)}")

Processed 50/300
Processed 100/300
Processed 150/300
Processed 200/300
Processed 250/300
Processed 300/300


In [45]:
import pandas as pd

df = pd.DataFrame(evaluation_results)

metrics = {
    "Recall@5": df["recall@5"].mean(),
    "Recall@10": df["recall@10"].mean(),
    "MRR@10": df["mrr@10"].mean(),
    "nDCG@10": df["ndcg@10"].mean(),
}

metrics

{'Recall@5': np.float64(0.7459444444444444),
 'Recall@10': np.float64(0.8088333333333334),
 'MRR@10': np.float64(0.6470886243386242),
 'nDCG@10': np.float64(0.6830483645447563)}

<h1> Hybrid Collection

In [46]:
from qdrant_client import models

HYBRID_COLLECTION = "scifact_hybrid"

if client.collection_exists(HYBRID_COLLECTION):
    client.delete_collection(HYBRID_COLLECTION)

client.create_collection(
    collection_name=HYBRID_COLLECTION,
    vectors_config={
        "dense": models.VectorParams(
            size=384,
            distance=models.Distance.COSINE,
        )
    },
    sparse_vectors_config={
        "bm25": models.SparseVectorParams(
            modifier=models.Modifier.IDF
        )
    },
)

print("Hybrid collection created")

Hybrid collection created


In [47]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6489.80it/s]


In [48]:
import uuid
from qdrant_client.models import PointStruct, Document

BATCH_SIZE = 100
items = list(corpus.items())

for start in range(0, len(items), BATCH_SIZE):

    batch = items[start:start + BATCH_SIZE]

    texts = [
        doc["title"] + "\n" + doc["text"]
        for _, doc in batch
    ]

    # Dense embeddings
    dense_embeddings = embedding_model.encode(
        texts,
        normalize_embeddings=True,
        show_progress_bar=False,
    )

    points = []

    for (doc_id, doc), dense_embedding, text in zip(
        batch,
        dense_embeddings,
        texts,
    ):
        points.append(
            PointStruct(
                id=str(uuid.uuid4()),

                vector={
                    "dense": dense_embedding.tolist(),

                    "bm25": Document(
                        text=text,
                        model="qdrant/bm25",
                    ),
                },

                payload={
                    "document_id": doc_id,
                    "title": doc["title"],
                    "text": doc["text"],
                },
            )
        )

    client.upsert(
        collection_name=HYBRID_COLLECTION,
        points=points,
    )

    print(
        f"Uploaded "
        f"{min(start + BATCH_SIZE, len(items))}/{len(items)}"
    )

Uploaded 100/5183
Uploaded 200/5183
Uploaded 300/5183
Uploaded 400/5183
Uploaded 500/5183
Uploaded 600/5183
Uploaded 700/5183
Uploaded 800/5183
Uploaded 900/5183
Uploaded 1000/5183
Uploaded 1100/5183
Uploaded 1200/5183
Uploaded 1300/5183
Uploaded 1400/5183
Uploaded 1500/5183
Uploaded 1600/5183
Uploaded 1700/5183
Uploaded 1800/5183
Uploaded 1900/5183
Uploaded 2000/5183
Uploaded 2100/5183
Uploaded 2200/5183
Uploaded 2300/5183
Uploaded 2400/5183
Uploaded 2500/5183
Uploaded 2600/5183
Uploaded 2700/5183
Uploaded 2800/5183
Uploaded 2900/5183
Uploaded 3000/5183
Uploaded 3100/5183
Uploaded 3200/5183
Uploaded 3300/5183
Uploaded 3400/5183
Uploaded 3500/5183
Uploaded 3600/5183
Uploaded 3700/5183
Uploaded 3800/5183
Uploaded 3900/5183
Uploaded 4000/5183
Uploaded 4100/5183
Uploaded 4200/5183
Uploaded 4300/5183
Uploaded 4400/5183
Uploaded 4500/5183
Uploaded 4600/5183
Uploaded 4700/5183
Uploaded 4800/5183
Uploaded 4900/5183
Uploaded 5000/5183
Uploaded 5100/5183
Uploaded 5183/5183


In [49]:
client.count(
    collection_name=HYBRID_COLLECTION
).count

5183

In [51]:
def dense_search(query_text, top_k=10):

    query_embedding = embedding_model.encode(
        query_text,
        normalize_embeddings=True,
    )

    results = client.query_points(
        collection_name=HYBRID_COLLECTION,
        query=query_embedding.tolist(),
        using="dense",
        limit=top_k,
        with_payload=True,
    ).points

    return [
        {
            "document_id": result.payload["document_id"],
            "score": result.score,
        }
        for result in results
    ]

In [52]:
dense_results = []

for i, (query_id, query_text) in enumerate(queries.items()):

    relevant_docs = [
        doc_id
        for doc_id, relevance in qrels.get(query_id, {}).items()
        if relevance > 0
    ]

    if not relevant_docs:
        continue

    results = dense_search(query_text, top_k=10)

    retrieved_docs = [
        result["document_id"]
        for result in results
    ]

    dense_results.append({
        "query_id": query_id,

        "recall@5": recall_at_k(
            retrieved_docs,
            relevant_docs,
            5,
        ),

        "recall@10": recall_at_k(
            retrieved_docs,
            relevant_docs,
            10,
        ),

        "mrr@10": reciprocal_rank(
            retrieved_docs,
            relevant_docs,
            10,
        ),

        "ndcg@10": ndcg_at_k(
            retrieved_docs,
            relevant_docs,
            10,
        ),
    })

    if (i + 1) % 50 == 0:
        print(f"Processed {i + 1}/{len(queries)}")

Processed 50/300
Processed 100/300
Processed 150/300
Processed 200/300
Processed 250/300
Processed 300/300


In [53]:
dense_df = pd.DataFrame(dense_results)

dense_metrics = {
    "Recall@5": dense_df["recall@5"].mean(),
    "Recall@10": dense_df["recall@10"].mean(),
    "MRR@10": dense_df["mrr@10"].mean(),
    "nDCG@10": dense_df["ndcg@10"].mean(),
}

dense_metrics

{'Recall@5': np.float64(0.7379444444444444),
 'Recall@10': np.float64(0.7833333333333333),
 'MRR@10': np.float64(0.6047248677248678),
 'nDCG@10': np.float64(0.6450816521455776)}

<h1> hybrid retrieval </h1>

In [54]:
def hybrid_search(query_text, top_k=10, candidate_k=20):

    # Dense
    query_embedding = embedding_model.encode(
        query_text,
        normalize_embeddings=True,
    )

    dense_results = client.query_points(
        collection_name=HYBRID_COLLECTION,
        query=query_embedding.tolist(),
        using="dense",
        limit=candidate_k,
        with_payload=True,
    ).points

    # BM25
    bm25_results = client.query_points(
        collection_name=HYBRID_COLLECTION,
        query=Document(
            text=query_text,
            model="qdrant/bm25",
        ),
        using="bm25",
        limit=candidate_k,
        with_payload=True,
    ).points

    return dense_results, bm25_results

In [ ]:
def rrf_fusion(dense_results, bm25_results, k=60, top_k=10):

    scores = {}
    payloads = {}

    for rank, result in enumerate(dense_results, start=1):

        doc_id = result.payload["document_id"]

        scores[doc_id] = scores.get(doc_id, 0) + (
            1 / (k + rank)
        )

        payloads[doc_id] = result.payload

    for rank, result in enumerate(bm25_results, start=1):

        doc_id = result.payload["document_id"]

        scores[doc_id] = scores.get(doc_id, 0) + (
            1 / (k + rank)
        )

        payloads[doc_id] = result.payload

    ranked = sorted(
        scores.items(),
        key=lambda x: x[1],
        reverse=True,
    )

    return [
        {
            "document_id": doc_id,
            "score": score,
            "title": payloads[doc_id]["title"],
            "text": payloads[doc_id]["text"],
        }
        for doc_id, score in ranked[:top_k]
    ]

In [56]:
dense_results, bm25_results = hybrid_search(query_text)

hybrid_results = rrf_fusion(
    dense_results,
    bm25_results,
)

for result in hybrid_results:
    print(result)

{'document_id': '7506409', 'score': 0.028814262023217248, 'title': 'Senescent growth arrest in mesenchymal stem cells is bypassed by Wip1-mediated downregulation of intrinsic stress signaling pathways.'}
{'document_id': '31803596', 'score': 0.02819138376017471, 'title': 'Gene expression analysis by cDNA microarray in oral squamous cell carcinoma.'}
{'document_id': '19343151', 'score': 0.01639344262295082, 'title': 'p16INK4A is a robust in vivo biomarker of cellular aging in human skin.'}
{'document_id': '5492542', 'score': 0.01639344262295082, 'title': 'Oral ciclopirox olamine displays biological activity in a phase I study in patients with advanced hematologic malignancies.'}
{'document_id': '18218379', 'score': 0.016129032258064516, 'title': 'Quantitative analysis of tumor-derived methylated p16INK4a sequences in plasma, serum, and blood cells of hepatocellular carcinoma patients.'}
{'document_id': '22309946', 'score': 0.016129032258064516, 'title': 'Asymptomatic spinal cord lesions 

In [57]:
def hybrid_search_final(query_text, candidate_k=20, top_k=10):

    dense_results, bm25_results = hybrid_search(
        query_text,
        candidate_k=candidate_k,
    )

    return rrf_fusion(
        dense_results,
        bm25_results,
        k=60,
        top_k=top_k,
    )

In [58]:
hybrid_results_eval = []

for i, (query_id, query_text) in enumerate(queries.items()):

    relevant_docs = [
        doc_id
        for doc_id, relevance in qrels.get(query_id, {}).items()
        if relevance > 0
    ]

    if not relevant_docs:
        continue

    results = hybrid_search_final(
        query_text,
        candidate_k=20,
        top_k=10,
    )

    retrieved_docs = [
        result["document_id"]
        for result in results
    ]

    hybrid_results_eval.append({
        "query_id": query_id,

        "recall@5": recall_at_k(
            retrieved_docs,
            relevant_docs,
            5,
        ),

        "recall@10": recall_at_k(
            retrieved_docs,
            relevant_docs,
            10,
        ),

        "mrr@10": reciprocal_rank(
            retrieved_docs,
            relevant_docs,
            10,
        ),

        "ndcg@10": ndcg_at_k(
            retrieved_docs,
            relevant_docs,
            10,
        ),
    })

    if (i + 1) % 50 == 0:
        print(f"Processed {i + 1}/{len(queries)}")

Processed 50/300
Processed 100/300
Processed 150/300
Processed 200/300
Processed 250/300
Processed 300/300


In [59]:
hybrid_df = pd.DataFrame(hybrid_results_eval)

hybrid_metrics = {
    "Recall@5": hybrid_df["recall@5"].mean(),
    "Recall@10": hybrid_df["recall@10"].mean(),
    "MRR@10": hybrid_df["mrr@10"].mean(),
    "nDCG@10": hybrid_df["ndcg@10"].mean(),
}

hybrid_metrics

{'Recall@5': np.float64(0.7854444444444445),
 'Recall@10': np.float64(0.8516666666666667),
 'MRR@10': np.float64(0.677910052910053),
 'nDCG@10': np.float64(0.714312463860793)}

<h1> cross encoder

In [60]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 4554.31it/s]


In [61]:
def rerank_results(query_text, candidates, top_k=10):

    pairs = [
        (
            query_text,
            candidate["text"]
        )
        for candidate in candidates
    ]

    scores = reranker.predict(pairs)

    reranked = []

    for candidate, score in zip(candidates, scores):

        reranked.append({
            **candidate,
            "rerank_score": float(score),
        })

    reranked.sort(
        key=lambda x: x["rerank_score"],
        reverse=True,
    )

    return reranked[:top_k]